<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/v3/modern-architectures/sliding-window-attention/sliding-window-attention.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("sliding-window-attention", ...)`


# Implement Sliding Window Attention

**Difficulty**: 🟡 Medium

**Companies**: Mistral, Anthropic, Google, DeepMind

---

### Problem Statement

Standard self-attention has **O(n²)** memory and compute complexity, which becomes prohibitive for long sequences. **Sliding window attention** (used in Mistral, Longformer, and others) restricts each token to only attend to a local window of **w** tokens on each side, reducing complexity to **O(n·w)**.

Your task is to implement sliding window attention by:
1. Creating a **band-diagonal attention mask** where token `i` can attend to tokens `[i-w, i+w]`.
2. Applying the mask to **scaled dot-product attention**.
3. Comparing the output with full attention to verify correctness.

---

### Requirements

1. **`create_sliding_window_mask(seq_len, window_size)`** — Returns a boolean mask of shape `(seq_len, seq_len)` where `mask[i][j] = True` if `|i - j| <= window_size`.
2. **`sliding_window_attention(q, k, v, window_size)`** — Applies scaled dot-product attention with the sliding window mask. Masked positions should get `-inf` before softmax.
3. **Memory comparison** — Show that sliding window uses O(n·w) non-zero entries vs O(n²) for full attention.

---

### Constraints

- ✅ Must handle batched inputs: `q, k, v` have shape `(batch_size, seq_len, d_model)`.
- ✅ When `window_size >= seq_len`, output must match full (unmasked) attention exactly.
- ✅ Attention weights for each query must sum to 1 (valid probability distribution).
- ❌ Do **not** use `torch.nn.MultiheadAttention` or any built-in attention layers.

---

<details>
  <summary>💡 Hint</summary>

  - Create the mask using: `mask = (torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)).abs() <= window_size`.
  - Apply the mask by setting `scores[~mask] = float('-inf')` before softmax.
  - Scale factor for attention: `1 / sqrt(d_model)`.
  - Non-zero entries in a band matrix of width 2w+1: approximately `n * (2w + 1)` (clamped at edges).

</details>

---

In [ ]:
import math
import torch
import torch.nn.functional as F

In [ ]:
# Setup
torch.manual_seed(42)

batch_size = 2
seq_len = 16
d_model = 32
window_size = 3  # attend to 3 tokens on each side

q = torch.randn(batch_size, seq_len, d_model)
k = torch.randn(batch_size, seq_len, d_model)
v = torch.randn(batch_size, seq_len, d_model)

print(f"Q/K/V shape: {q.shape}")
print(f"Window size: {window_size} (each token attends to {2 * window_size + 1} positions)")

In [ ]:
def create_sliding_window_mask(seq_len: int, window_size: int) -> torch.Tensor:
    """
    Create a sliding window attention mask.

    Args:
        seq_len:     sequence length
        window_size: number of tokens to attend to on each side

    Returns:
        Boolean tensor of shape (seq_len, seq_len) where True = allowed to attend.
    """
    # TODO: Create a matrix of pairwise distances |i - j|
    # TODO: Return boolean mask where distance <= window_size
    i = torch.arange(seq_len)
    j = torch.arange(seq_len)
    distance = (i.unsqueeze(1) - j.unsqueeze(0)).abs()
    mask = (distance <= window_size)
    return mask

def full_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Standard scaled dot-product attention (no mask).

    Args:
        q, k, v: (batch_size, seq_len, d_model)

    Returns:
        Output tensor of shape (batch_size, seq_len, d_model)
    """
    # TODO: Compute scaled dot-product attention
    B, seq_len, d_model = q.size()
    scores = (q @ k.transpose(-2, -1)) / torch.sqrt(torch.Tensor(d_model))
    weights = torch.softmax(scores, dim=-1)
    output = weights @ v
    return output


def sliding_window_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                              window_size: int) -> torch.Tensor:
    """
    Scaled dot-product attention with sliding window mask.

    Args:
        q, k, v:     (batch_size, seq_len, d_model)
        window_size: number of tokens to attend to on each side

    Returns:
        Output tensor of shape (batch_size, seq_len, d_model)
    """
    # TODO: Step 1 - Compute raw attention scores: q @ k^T / sqrt(d_model)

    # TODO: Step 2 - Create the sliding window mask

    # TODO: Step 3 - Apply mask: set masked positions to -inf

    # TODO: Step 4 - Softmax and compute weighted sum with v

    ...


# Quick test
mask = create_sliding_window_mask(seq_len, window_size)
print(f"Mask shape: {mask.shape}")
print(f"Mask (first 8x8):\n{mask[:8, :8].int()}")

out = sliding_window_attention(q, k, v, window_size)
print(f"\nOutput shape: {out.shape}")

In [ ]:
# Validation

# Validation 1: When window_size >= seq_len, should match full attention
print("=== Full Window = Full Attention ===")
out_full = full_attention(q, k, v)
out_big_window = sliding_window_attention(q, k, v, window_size=seq_len)
assert torch.allclose(out_full, out_big_window, atol=1e-6), "Large window should match full attention!"
print("PASSED: window_size=seq_len matches full attention.\n")

# Validation 2: Attention weights sum to 1
print("=== Attention Weights Sum to 1 ===")
d = q.size(-1)
scores = q @ k.transpose(-2, -1) / math.sqrt(d)
mask = create_sliding_window_mask(seq_len, window_size)
scores = scores.masked_fill(~mask.unsqueeze(0), float('-inf'))
weights = F.softmax(scores, dim=-1)
weight_sums = weights.sum(dim=-1)
assert torch.allclose(weight_sums, torch.ones_like(weight_sums), atol=1e-6), "Weights must sum to 1!"
print("PASSED: All attention weight rows sum to 1.\n")

# Validation 3: Smaller window changes the output
print("=== Different Window Sizes Give Different Outputs ===")
out_w1 = sliding_window_attention(q, k, v, window_size=1)
out_w3 = sliding_window_attention(q, k, v, window_size=3)
assert not torch.allclose(out_w1, out_w3), "Different window sizes should give different outputs!"
print("PASSED: window_size=1 and window_size=3 produce different outputs.\n")

# Validation 4: Memory comparison
print("=== Memory Comparison ===")
for n in [128, 512, 2048, 8192]:
    w = 64
    full_entries = n * n
    m = create_sliding_window_mask(n, w)
    window_entries = m.sum().item()
    ratio = window_entries / full_entries * 100
    print(f"  seq_len={n:5d}, window={w}: {window_entries:>10,.0f} / {full_entries:>12,.0f} entries ({ratio:.1f}%)")
print("PASSED: Sliding window uses far fewer entries for long sequences.\n")

print("All tests passed!")

---

# Bonus: Efficient Sliding Window Attention with `unfold`

**Difficulty**: 🟡 Medium

---

### Problem Statement

The mask-based implementation above still computes the **full n×n** score matrix, then throws away most of it. For truly long sequences this defeats the purpose of windowed attention.

Your goal is to implement sliding window attention that **never builds the n×n matrix** — achieving genuine O(n·w) compute and memory using `torch.Tensor.unfold`.

---

### Background: `torch.Tensor.unfold`

`unfold(dim, size, step)` slides a window along a dimension and exposes the window contents as a **new last dimension**. It returns a **view** (no data copy).

```python
x = torch.arange(10)                # shape (10,)
x.unfold(0, 4, 1)                   # shape (7, 4) — 7 windows of size 4
# tensor([[0, 1, 2, 3],
#          [1, 2, 3, 4],
#          [2, 3, 4, 5],
#          ...])
```

Applied to a `(batch, seq_len, d)` tensor along `dim=1`:
- `k.unfold(1, window, 1)` → `(batch, num_windows, d, window)` — each position gets its own slice of neighbors. The key insight is that overlapping windows **share memory** via strides.

---

### Requirements

1. **`sliding_window_attention_unfold(q, k, v, window_size)`**
   - Pad `k` and `v` with zeros on both sides by `window_size` along the sequence dimension.
   - Use `unfold` to extract all windows at once — no Python loops.
   - Compute per-position attention scores of shape `(batch, seq_len, 1, 2w+1)` instead of `(batch, seq_len, seq_len)`.
   - Apply softmax and compute the weighted sum.

2. **Verify correctness** — output must match the mask-based version exactly.

3. **Compare memory** — count the number of score entries computed: `n × (2w+1)` vs `n × n`.

---

### Constraints

- ✅ No Python `for` loops over sequence positions.
- ✅ Must handle batched inputs.
- ✅ Must match the mask-based implementation output exactly (within float tolerance).
- ❌ Do **not** build an `(seq_len, seq_len)` score matrix.

---

<details>
  <summary>💡 Hint</summary>

  - `F.pad(k, (0, 0, w, w))` pads the sequence dimension (dim=1) on both sides with zeros. The pad args are `(last_dim_left, last_dim_right, second_last_dim_left, second_last_dim_right)`.
  - After padding, `k_pad.unfold(1, 2*w+1, 1)` gives shape `(batch, seq_len, d, 2w+1)`. Permute to get `(batch, seq_len, 2w+1, d)`.
  - Use `q.unsqueeze(2)` to get `(batch, seq_len, 1, d)` so the matmul broadcasts correctly.
  - The padded zeros will get `-inf` scores after masking or naturally low scores — but to be exact, mask out pad positions before softmax.

</details>

---

In [ ]:
def sliding_window_attention_unfold(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                                    window_size: int) -> torch.Tensor:
    """
    Efficient sliding window attention using unfold — O(n·w) compute, no n×n matrix.

    Args:
        q, k, v:     (batch_size, seq_len, d_model)
        window_size: number of tokens to attend to on each side

    Returns:
        Output tensor of shape (batch_size, seq_len, d_model)
    """
    batch, seq_len, d = q.shape
    w = window_size
    win = 2 * w + 1

    # TODO: Step 1 - Pad k and v along the sequence dimension (dim=1) by w on each side
    # Use F.pad — remember pad args go from last dim to first: (d_left, d_right, seq_left, seq_right)

    # TODO: Step 2 - Use .unfold(1, win, 1) to extract sliding windows
    # Result shape after unfold: (batch, seq_len, d, win)
    # Permute to get: (batch, seq_len, win, d)

    # TODO: Step 3 - Compute attention scores
    # q: (batch, seq_len, d) → unsqueeze to (batch, seq_len, 1, d)
    # scores = q_expanded @ k_windows.transpose(-2, -1) → (batch, seq_len, 1, win)
    # Scale by sqrt(d)

    # TODO: Step 4 - Mask out padded positions (they are all zeros from F.pad)
    # Create a 1-tensor, pad and unfold the same way to find which positions are real vs padded
    # Set padded positions to -inf before softmax

    # TODO: Step 5 - Softmax over the window dimension and compute weighted sum with v_windows

    ...

In [ ]:
# Bonus Validation

# Test 1: Must match mask-based implementation
print("=== Unfold vs Mask-Based ===")
out_mask = sliding_window_attention(q, k, v, window_size)
out_unfold = sliding_window_attention_unfold(q, k, v, window_size)
assert torch.allclose(out_mask, out_unfold, atol=1e-5), \
    f"Outputs differ! Max diff: {(out_mask - out_unfold).abs().max():.2e}"
print("PASSED: unfold version matches mask-based version.\n")

# Test 2: Full window matches full attention
print("=== Unfold Full Window = Full Attention ===")
out_full = full_attention(q, k, v)
out_unfold_full = sliding_window_attention_unfold(q, k, v, window_size=seq_len)
assert torch.allclose(out_full, out_unfold_full, atol=1e-5), "Full window should match full attention!"
print("PASSED: window_size=seq_len matches full attention.\n")

# Test 3: Memory comparison — score entries computed
print("=== Score Entries Computed ===")
for n in [128, 512, 2048, 8192]:
    w = 64
    full_scores = n * n
    unfold_scores = n * (2 * w + 1)
    ratio = unfold_scores / full_scores * 100
    print(f"  seq_len={n:5d}, window={w}: unfold={unfold_scores:>10,} vs full={full_scores:>12,} ({ratio:.1f}%)")
print("PASSED: unfold computes far fewer score entries.\n")

print("All bonus tests passed!")